# 01 - Ingesta de datos

**Proyecto:** Analítica de ventas de videojuegos en Databricks  
**Autor:** Darren J. Blackman M.  
**Asignatura:** Análisis de Datos y Toma de Decisiones en Computación

## Objetivo
Cargar el conjunto de datos previamente subido a Databricks como la tabla **vgsales_raw**, revisar su estructura y conservar una copia inicial llamada **vgsales_bronze**.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

CATALOGO = spark.sql("SELECT current_catalog()").first()[0]
ESQUEMA = "default"

def tabla(nombre):
    return f"`{CATALOGO}`.`{ESQUEMA}`.`{nombre}`"

print(f"Catálogo activo: {CATALOGO}")
print(f"Esquema utilizado: {ESQUEMA}")

Catálogo activo: workspace
Esquema utilizado: default


In [0]:
TABLA_ORIGEN = tabla("vgsales_raw")
TABLA_BRONZE = tabla("vgsales_bronze")

try:
    df_raw = spark.table(TABLA_ORIGEN)
except Exception as exc:
    raise RuntimeError(
        "No se encontró la tabla vgsales_raw. Cargue datos/vgsales.csv desde la interfaz "
        "de Databricks y cree la tabla en el esquema default antes de continuar."
    ) from exc

print(f"Tabla encontrada: {TABLA_ORIGEN}")

Tabla encontrada: `workspace`.`default`.`vgsales_raw`


In [0]:
display(df_raw.limit(10))
print(f"Registros iniciales: {df_raw.count():,}")
print(f"Columnas: {len(df_raw.columns)}")
print(df_raw.columns)

Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
1,Wii Sports,Wii,2006,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
2,Super Mario Bros.,NES,1985,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
3,Mario Kart Wii,Wii,2008,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
4,Wii Sports Resort,Wii,2009,Sports,Nintendo,15.75,11.01,3.28,2.96,33.0
5,Pokemon Red/Pokemon Blue,GB,1996,Role-Playing,Nintendo,11.27,8.89,10.22,1.0,31.37
6,Tetris,GB,1989,Puzzle,Nintendo,23.2,2.26,4.22,0.58,30.26
7,New Super Mario Bros.,DS,2006,Platform,Nintendo,11.38,9.23,6.5,2.9,30.01
8,Wii Play,Wii,2006,Misc,Nintendo,14.03,9.2,2.93,2.85,29.02
9,New Super Mario Bros. Wii,Wii,2009,Platform,Nintendo,14.59,7.06,4.7,2.26,28.62
10,Duck Hunt,NES,1984,Shooter,Nintendo,26.93,0.63,0.28,0.47,28.31


Registros iniciales: 16,598
Columnas: 11
['Rank', 'Name', 'Platform', 'Year', 'Genre', 'Publisher', 'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']


In [0]:
# Esquema inferido por Databricks
df_raw.printSchema()

root
 |-- Rank: long (nullable = true)
 |-- Name: string (nullable = true)
 |-- Platform: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Publisher: string (nullable = true)
 |-- NA_Sales: double (nullable = true)
 |-- EU_Sales: double (nullable = true)
 |-- JP_Sales: double (nullable = true)
 |-- Other_Sales: double (nullable = true)
 |-- Global_Sales: double (nullable = true)



In [0]:
# Conteo de nulos por columna
nulos = df_raw.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_raw.columns
])
display(nulos)

Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,0,0,0,0,0,0,0,0,0,0


In [0]:
# Copia de la capa inicial (bronze)
(
    df_raw.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLA_BRONZE)
)

print(f"Tabla bronze creada: {TABLA_BRONZE}")

Tabla bronze creada: `workspace`.`default`.`vgsales_bronze`


In [0]:
# Verificación de persistencia
verificacion = spark.table(TABLA_BRONZE)
print(f"Registros guardados en bronze: {verificacion.count():,}")
display(verificacion.limit(5))

Registros guardados en bronze: 16,598


Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
1,Wii Sports,Wii,2006,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
2,Super Mario Bros.,NES,1985,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
3,Mario Kart Wii,Wii,2008,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
4,Wii Sports Resort,Wii,2009,Sports,Nintendo,15.75,11.01,3.28,2.96,33.0
5,Pokemon Red/Pokemon Blue,GB,1996,Role-Playing,Nintendo,11.27,8.89,10.22,1.0,31.37
